# FELIX hit sweep — every phenotype, every arm

Walks the whole run table, screens every genome-wide hit in all three arms, and puts the surviving candidates in one place.

**Why a sweep and not 64 single-phenotype reports.** Two questions only answerable across phenotypes:

- *Does a locus recur?* A site that surfaces as a hit in several unrelated phenotypes is usually a property of the locus — mapping, genotyping, a degenerate dosage distribution — rather than real pleiotropy. Recurrence is a screen in its own right.
- *How does the artifact rate scale?* The SPA fallback bites hardest at low case counts. Plotting hit count against case count over the full set shows where the repeat arm stops being usable, which is what a case floor has to be set from.

**Cost.** Hit tables for the repeat arms exist only inside `run_summary_tar` (the table's `top_hits_tables` column points at the main arm alone). The sweep streams each bundle through `tar` and keeps only `calibration_summary.tsv` and `top_hits.tsv` — about 0.2 MB per run instead of 125 MB, and the tarball never lands on disk. Harvesting is cached per run, so a second pass is free.

**Positions, not marker IDs.** The SNV arm writes `chr9_8991058_T_TA;…` where the repeat arms write `9-8991058-8991069-A` for the same site, so cross-arm and cross-phenotype matching is done by clustering on position.

## 1 · Setup

In [ ]:
import os, json, subprocess, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

ETYPE    = "flare_lai_prod_set"
ID_COL   = f"entity:{ETYPE}_id"
TAR_COL  = "run_summary_tar"
NUM_ANCS = 5
STORE    = "harvest_cache"      # small per-run extracts; delete to re-harvest

WS_NS   = os.environ.get("WORKSPACE_NAMESPACE")
WS_NAME = os.environ.get("WORKSPACE_NAME")
BUCKET  = os.environ.get("WORKSPACE_BUCKET")
print("workspace:", WS_NS, "/", WS_NAME)
assert WS_NS and WS_NAME, "set WS_NS / WS_NAME by hand if the env vars are absent"

# One canonical copy of the analysis module: felix/scripts/, synced to the
# bucket by 00_sync_repo. Pull it rather than embedding a second copy here.
MODULE = Path("felix_phenotype_report.py")
if not MODULE.exists():
    src = f"{BUCKET}/felix/scripts/felix_phenotype_report.py"
    subprocess.run(["gsutil", "cp", src, str(MODULE)], check=True)
    print("fetched", src)
import importlib, felix_phenotype_report as fpr
importlib.reload(fpr)
print("module ok:", hasattr(fpr, "sweep"))

## 2 · Run table

In [ ]:
from firecloud import api as fapi

def entities_to_df(ns, name, etype):
    r = fapi.get_entities(ns, name, etype); r.raise_for_status()
    rows = []
    for e in r.json():
        rec = {f"entity:{etype}_id": e["name"]}
        for k, v in e.get("attributes", {}).items():
            if isinstance(v, dict) and "items" in v:
                v = list(v["items"])
            rec[k] = v
        rows.append(rec)
    return pd.DataFrame(rows)

runs = entities_to_df(WS_NS, WS_NAME, ETYPE).sort_values(ID_COL).reset_index(drop=True)
done = runs[runs[TAR_COL].notna()].copy()
print(f"{len(runs)} rows in {ETYPE}; {len(done)} have {TAR_COL}")

## 3 · Sweep every run

In [ ]:
%%time
cal, hits, failures = fpr.sweep(done, ID_COL, TAR_COL, store=STORE,
                                num_ancs=NUM_ANCS, workers=8)
print(f"\n{len(cal)} run x arm calibration rows; {len(hits)} genome-wide hits; "
      f"{len(failures)} harvest failures")
if len(failures):
    display(failures)
size_mb = sum(f.stat().st_size for f in Path(STORE).rglob("*") if f.is_file()) / 1e6
print(f"harvest cache: {size_mb:.1f} MB")

## 4 · Per-phenotype summary

In [ ]:
# Per-phenotype summary: one row per run, arms side by side
wide = cal.pivot_table(index=["run", "n_cases", "N_eff"], columns="arm",
                       values=["lambda_gc", "n_gw_sig"], aggfunc="first").reset_index()
wide.columns = [c[0] if not c[1] else f"{c[0]}_{c[1]}" for c in wide.columns]

if len(hits):
    kept = (hits[hits.verdict == "keep"].groupby(["run", "arm"]).size()
            .unstack(fill_value=0).add_prefix("kept_").reset_index())
    wide = wide.merge(kept, on="run", how="left")
for c in [c for c in wide.columns if c.startswith("kept_")]:
    wide[c] = wide[c].fillna(0).astype(int)

wide = wide.sort_values("n_cases")
display(Markdown(f"### {len(wide)} runs, sorted by case count"))
display(wide)

tot_rep = int(cal.n_gw_sig.sum())
tot_keep = int((hits.verdict == "keep").sum()) if len(hits) else 0
tot_art = int((hits.verdict == "ARTIFACT").sum()) if len(hits) else 0
display(Markdown(
    f"**{tot_rep} hits reported across all runs and arms → {tot_art} flagged as "
    f"SPA fallbacks, {tot_keep} retained.** "
    f"({tot_rep - tot_art - tot_keep} reported hits fall outside the bundles' "
    f"top-50 files and could not be screened.)"))

## 5 · Cross-phenotype views

In [ ]:
display(Markdown("### Loci hit in more than one phenotype"))
rec = fpr.recurrence(hits, min_phenotypes=2)
if len(rec):
    display(rec.head(40))
    display(Markdown("> A locus recurring across unrelated phenotypes is more often a "
                     "property of the locus than real pleiotropy. Check these before "
                     "treating any of them as findings."))
else:
    display(Markdown("None — no locus is a hit in two or more phenotypes."))

display(Markdown("### Retained hits corroborated across arms (same phenotype)"))
cor = fpr.corroborated(hits)
if len(cor):
    display(cor)
    display(Markdown("> Independent codings of the same data agreeing is the strongest "
                     "evidence available short of replication."))
else:
    display(Markdown("None."))

display(Markdown("### Every retained hit"))
if len(hits) and (hits.verdict == "keep").any():
    display(hits[hits.verdict == "keep"].sort_values("p")
            [["run", "phenotype", "arm", "CHR", "POS", "MarkerID", "p", "case_carriers"]])
else:
    display(Markdown("None retained."))

## 6 · Scaling with case count

In [ ]:
ru = cal[cal.arm == "ru"].copy()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))

ax[0].scatter(ru.n_cases, ru.lambda_gc, s=20, color="#3b6ea5")
ax[0].axhline(1.0, ls="--", lw=0.8, color="#888888")
ax[0].set_xscale("log"); ax[0].set_xlabel("cases")
ax[0].set_ylabel(r"$\lambda$"); ax[0].set_title("Calibration vs case count (ru arm)")

ax[1].scatter(ru.n_cases, ru.n_gw_sig + 0.5, s=20, color="#d1495b")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("cases"); ax[1].set_ylabel("reported hits + 0.5")
ax[1].set_title("Hits before screening (ru arm)")

if len(hits):
    per = (hits.groupby(["run", "arm"]).verdict
             .value_counts().unstack(fill_value=0).reset_index())
    per = per[per.arm == "ru"].merge(ru[["run", "n_cases"]], on="run", how="left")
    if "ARTIFACT" in per.columns:
        ax[2].scatter(per.n_cases, per["ARTIFACT"] + 0.5, s=20,
                      color="#d1495b", label="flagged")
    if "keep" in per.columns:
        ax[2].scatter(per.n_cases, per["keep"] + 0.5, s=28, facecolors="none",
                      edgecolors="#2e7d5b", label="retained")
    ax[2].set_xscale("log"); ax[2].set_yscale("log")
    ax[2].set_xlabel("cases"); ax[2].set_ylabel("hits + 0.5")
    ax[2].set_title("After screening (ru arm)"); ax[2].legend(frameon=False)
fig.tight_layout(); plt.show()

## 7 · Export

In [ ]:
cal.to_csv("felix_sweep_calibration.tsv", sep="\t", index=False)
wide.to_csv("felix_sweep_by_phenotype.tsv", sep="\t", index=False)
if len(hits):
    hits.to_csv("felix_sweep_all_hits.tsv", sep="\t", index=False)
    fpr.recurrence(hits).to_csv("felix_sweep_recurrent_loci.tsv", sep="\t", index=False)
    fpr.corroborated(hits).to_csv("felix_sweep_corroborated.tsv", sep="\t", index=False)
print("wrote:", sorted(p.name for p in Path(".").glob("felix_sweep_*.tsv")))
if BUCKET:
    print(f"copy up with:  gsutil -m cp felix_sweep_*.tsv {BUCKET}/analysis/")

## Caveats

- **Only hits inside each bundle's top-50-per-arm file are screened.** A run reporting more than 50 hits in an arm has the remainder unscreened; the summary counts them so the gap is visible.
- **Retained ≠ real.** It means no detectable SPA fallback and a plausible case-carrier count. Cross-arm corroboration and recurrence are the next filters, and neither substitutes for replication.
- **λ is untouched by the screen.** The per-phenotype table reports λ as FELIX computed it. A run with λ ≈ 0.6 and zero retained hits is still a run whose repeat arm is mis-calibrated.
- Delete `harvest_cache/` to force a re-harvest after a new batch of runs completes.